# 라라 vs 진힐라 — E2E 마일스톤 통합 노트북

M1 정확성 검증 → M2 tensor batch 환경 → M3 반복 rollout 학습 → M4 stale-actor V-trace → M5 SPS 프로파일링을 같은 Colab 런타임에서 순차 실행한다.

> 원본 baseline 셀은 보존한다. 아다 E2E 셀이 수정된 환경과 학습 경로를 실행한다.

In [ ]:
from __future__ import annotations
import random
from dataclasses import dataclass, field
from enum import IntEnum
import torch
import torch.nn as nn
from torch.distributions import Categorical

class ActionType(IntEnum):
    SKILL = 0
    EVADE = 1
    WAIT = 2
    HARVEST = 3

class EntityField(IntEnum):
    TYPE_ID = 0
    X = 1
    Y = 2
    W = 3
    H = 4
    VX = 5
    VY = 6
    TTL = 7
    DMG = 8
    IS_PLATFORM = 9
    REACHABLE = 10
    SAFETY = 11

class ScalarIndex(IntEnum):
    PREV_INTERRUPT = 0
    PREV_ELAPSED_NORM = 1
    HAZARD_LEVEL = 2
    BOSS_HP_NORM = 3
    GREEN_SKULLS = 4
    RED_SKULLS = 5
    SOUL_SPLIT_ETA_NORM = 6
    POTION_UNLOCK_REMAIN = 7
    PLAYER_HP_NORM = 8
    PHASE = 9
    ALTAR_PRESENT = 10
    CANDLES = 11
    TIME_REMAIN_NORM = 12

MAX_ENTITIES = 64
ENTITY_DIM = 12
SCALAR_DIM = 32
NUM_SKILLS = 12
DELAY_BINS = 48
LSTM_HIDDEN = 128
LSTM_LAYERS = 2
NULL_ENTITY_INDEX = 0
STAY_ANCHOR_INDEX = 1
TICKS_PER_SEC = 30
EPISODE_MINUTES = 20
EPISODE_TICKS = EPISODE_MINUTES * 60 * TICKS_PER_SEC
THREAD_TELEGRAPH = int(1.32 * TICKS_PER_SEC)
SOUL_ABSORB_TICKS = 3 * TICKS_PER_SEC
HARVEST_HITS = 10
LANE_X = [0., 2., 4., 6., 8., 10., 12.]

class ContextGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x, ctx):
        return self.proj(x * torch.sigmoid(self.gate(ctx)))

class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.scalar = nn.Sequential(nn.Linear(SCALAR_DIM, 64), nn.ReLU())
        self.ctx = nn.Linear(SCALAR_DIM, LSTM_HIDDEN)
        self.entity = nn.Linear(ENTITY_DIM, 64)
        self.prev_type = nn.Embedding(4, 16)
        self.prev_delay = nn.Embedding(DELAY_BINS, 16)
        self.prev_elapsed = nn.Embedding(DELAY_BINS, 16)
        self.lstm = nn.LSTM(64 + 64 + 48, LSTM_HIDDEN, LSTM_LAYERS, batch_first=True)
        self.glu = ContextGLU(LSTM_HIDDEN)
        self.type_head = nn.Linear(LSTM_HIDDEN, 4)
        self.delay_head = nn.Linear(LSTM_HIDDEN, DELAY_BINS)
        self.skill_head = nn.Linear(LSTM_HIDDEN, NUM_SKILLS)
        self.ptr_query = nn.Linear(LSTM_HIDDEN, 64)
        self.value_head = nn.Linear(LSTM_HIDDEN, 1)

    def forward(self, scalars, entities, entity_mask, prev_type, prev_delay, prev_elapsed, hidden=None, gt=None):
        B, T, N, _ = entities.shape
        e = self.entity(entities)
        m = entity_mask.float().unsqueeze(-1)
        pooled = (e * m).sum(2) / m.sum(2).clamp_min(1.0)
        x = torch.cat([self.scalar(scalars), pooled, self.prev_type(prev_type), self.prev_delay(prev_delay), self.prev_elapsed(prev_elapsed)], -1)
        core, h = self.lstm(x, hidden)
        value = self.value_head(core).squeeze(-1)
        z = self.glu(core, self.ctx(scalars))
        typelogits = self.type_head(z)
        altar = scalars[..., ScalarIndex.ALTAR_PRESENT] > 0.5
        typelogits[..., ActionType.HARVEST] = typelogits[..., ActionType.HARVEST].masked_fill(~altar, -1e9)
        dist_t = Categorical(logits=typelogits)
        a_t = gt['type'] if gt is not None else dist_t.sample()
        delaylogits = self.delay_head(z)
        dist_d = Categorical(logits=delaylogits)
        a_d = gt['delay'] if gt is not None else dist_d.sample()
        skilllogits = self.skill_head(z)
        dist_s = Categorical(logits=skilllogits)
        a_s = gt['skill'] if gt is not None else dist_s.sample()
        keys = e
        q = self.ptr_query(z).unsqueeze(2)
        ptrlogits = torch.matmul(q, keys.transpose(-1, -2)).squeeze(2) / (64 ** 0.5)
        valid_ptr = entity_mask.clone()
        valid_ptr[..., NULL_ENTITY_INDEX] = False
        ptrlogits = ptrlogits.masked_fill(~valid_ptr, -1e9)
        dist_p = Categorical(logits=ptrlogits)
        a_p = gt['pointer'] if gt is not None else dist_p.sample()
        is_skill = (a_t == ActionType.SKILL).float()
        is_evade = (a_t == ActionType.EVADE).float()
        logp = dist_t.log_prob(a_t) + dist_d.log_prob(a_d) + is_skill * dist_s.log_prob(a_s) + is_evade * dist_p.log_prob(a_p)
        entropy = dist_t.entropy() + dist_d.entropy() + is_skill * dist_s.entropy() + is_evade * dist_p.entropy()
        return {'type': a_t, 'delay': a_d, 'skill': a_s, 'pointer': a_p, 'logp': logp, 'entropy': entropy, 'value': value, 'hidden': h}

@dataclass
class JinHillaEnv:
    rng: random.Random = field(default_factory=lambda: random.Random(7))
    def reset(self):
        self.t = 0
        self.boss_hp = 1.0
        self.green = 5
        self.red = 0
        self.lane = 3
        self.altar_lane = None
        self.harvest = 0
        self.stun = 0
        self.thread = [0] * 7
        self.scythe = 150 * TICKS_PER_SEC
        self.pending = (ActionType.WAIT, 0)
        return self.obs()
    def obs(self):
        entities = torch.zeros(MAX_ENTITIES, ENTITY_DIM)
        mask = torch.zeros(MAX_ENTITIES, dtype=torch.bool)
        mask[NULL_ENTITY_INDEX] = True
        mask[STAY_ANCHOR_INDEX] = True
        entities[STAY_ANCHOR_INDEX, EntityField.X] = LANE_X[self.lane]
        entities[STAY_ANCHOR_INDEX, EntityField.IS_PLATFORM] = 1.
        entities[STAY_ANCHOR_INDEX, EntityField.REACHABLE] = 1.
        for i in range(5):
            slot = i + 2
            entities[slot, EntityField.X] = LANE_X[i]
            entities[slot, EntityField.IS_PLATFORM] = 1.
            entities[slot, EntityField.REACHABLE] = 1.
            mask[slot] = True
        if self.altar_lane is not None:
            entities[7, EntityField.X] = LANE_X[self.altar_lane]
            entities[7, EntityField.IS_PLATFORM] = 1.
            entities[7, EntityField.REACHABLE] = 1.
            mask[7] = True
        s = torch.zeros(SCALAR_DIM)
        s[ScalarIndex.BOSS_HP_NORM] = self.boss_hp
        s[ScalarIndex.GREEN_SKULLS] = self.green / 5.
        s[ScalarIndex.RED_SKULLS] = self.red / 5.
        s[ScalarIndex.SOUL_SPLIT_ETA_NORM] = min(1., self.scythe / (180 * TICKS_PER_SEC))
        s[ScalarIndex.ALTAR_PRESENT] = float(self.altar_lane is not None)
        s[ScalarIndex.TIME_REMAIN_NORM] = max(0., 1. - self.t / EPISODE_TICKS)
        return {'scalars': s, 'entities': entities, 'entity_mask': mask}
    def step(self, action_type, delay, skill, pointer):
        self.pending = (int(action_type), int(pointer))
        reward = 0.
        elapsed = 0
        interrupted = False
        terminated = False
        for _ in range(int(delay) + 1):
            a, p = self.pending
            if self.stun > 0:
                self.stun -= 1
            elif a == ActionType.EVADE and 1 <= p <= 6:
                self.lane = p - 1
            elif a == ActionType.SKILL:
                self.boss_hp = max(0., self.boss_hp - 0.002)
                reward += 0.016
            elif a == ActionType.HARVEST and self.altar_lane is not None and abs(self.lane - self.altar_lane) <= 1:
                self.harvest += 1
                if self.harvest >= HARVEST_HITS:
                    reward += 0.6 + 0.1 * self.red
                    self.green = min(5, self.green + self.red)
                    self.red = 0
                    self.altar_lane = None
                    self.harvest = 0
            self.t += 1
            elapsed += 1
            for i in range(7):
                self.thread[i] = max(0, self.thread[i] - 1)
            if self.t % (8 * TICKS_PER_SEC) == 1:
                for i in range(7): self.thread[i] = THREAD_TELEGRAPH + i * 3
            if 0 < self.thread[self.lane] <= THREAD_TELEGRAPH and self.stun == 0:
                self.green -= 1
                self.red += 1
                self.stun = SOUL_ABSORB_TICKS
                reward -= 0.15
                interrupted = True
                if self.red >= 3 and self.altar_lane is None and self.green > 0:
                    self.altar_lane = self.rng.choice([1,2,3,4,5])
                if self.green <= 0:
                    terminated = True
                    reward -= 10.
                    break
            self.scythe -= 1
            if self.scythe <= 0:
                reward -= 0.8 * self.red
                self.red = 0
                self.scythe = 152 * TICKS_PER_SEC
            if self.boss_hp <= 0:
                terminated = True
                reward += 10.
                break
        truncated = self.t >= EPISODE_TICKS and not terminated
        return self.obs(), reward, terminated, truncated, elapsed, interrupted

def vtrace(log_rhos, discounts, rewards, values, bootstrap):
    with torch.no_grad():
        rho = torch.exp(log_rhos).clamp(max=1.0)
        next_values = torch.cat([values[1:], bootstrap[None]], 0)
        delta = rho * (rewards + discounts * next_values - values)
        acc = torch.zeros_like(bootstrap)
        corrections = []
        for t in reversed(range(len(discounts))):
            acc = delta[t] + discounts[t] * rho[t] * acc
            corrections.append(acc)
        corrections.reverse()
        vs = values + torch.stack(corrections)
        vs_next = torch.cat([vs[1:], bootstrap[None]], 0)
        adv = rho * (rewards + discounts * vs_next - values)
    return vs, adv

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
policy = Policy().to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
env = JinHillaEnv()
obs = env.reset()
hidden = None
prev_type = torch.tensor([[ActionType.WAIT]], device=device)
prev_delay = torch.zeros((1,1), dtype=torch.long, device=device)
prev_elapsed = torch.zeros((1,1), dtype=torch.long, device=device)
records = []
for _ in range(8):
    with torch.no_grad():
        out = policy(obs['scalars'][None,None].to(device), obs['entities'][None,None].to(device), obs['entity_mask'][None,None].to(device), prev_type, prev_delay, prev_elapsed, hidden)
    a_t = int(out['type'].item())
    a_d = int(out['delay'].item())
    a_s = int(out['skill'].item())
    a_p = int(out['pointer'].item())
    next_obs, reward, terminated, truncated, elapsed, interrupted = env.step(a_t, a_d, a_s, a_p)
    records.append((obs, a_t, a_d, a_s, a_p, float(out['logp'].item()), reward, 0. if terminated else 0.999 ** elapsed))
    obs = next_obs
    hidden = tuple(x.detach() for x in out['hidden'])
    prev_type = torch.tensor([[a_t]], device=device)
    prev_delay = torch.tensor([[a_d]], device=device)
    prev_elapsed = torch.tensor([[min(max(elapsed-1,0), DELAY_BINS-1)]], device=device)
    if terminated or truncated: break
with torch.no_grad():
    bootstrap = torch.tensor(0. if terminated else policy(obs['scalars'][None,None].to(device), obs['entities'][None,None].to(device), obs['entity_mask'][None,None].to(device), prev_type, prev_delay, prev_elapsed, hidden)['value'].item(), device=device)
T = len(records)
scalars = torch.stack([r[0]['scalars'] for r in records])[None].to(device)
entities = torch.stack([r[0]['entities'] for r in records])[None].to(device)
masks = torch.stack([r[0]['entity_mask'] for r in records])[None].to(device)
gt = {
    'type': torch.tensor([[r[1] for r in records]], device=device),
    'delay': torch.tensor([[r[2] for r in records]], device=device),
    'skill': torch.tensor([[r[3] for r in records]], device=device),
    'pointer': torch.tensor([[r[4] for r in records]], device=device),
}
pt = torch.cat([torch.tensor([[ActionType.WAIT]], device=device), gt['type'][:,:-1]], 1)
pd = torch.cat([torch.zeros((1,1), dtype=torch.long, device=device), gt['delay'][:,:-1]], 1)
pe = torch.zeros_like(pd)
out = policy(scalars, entities, masks, pt, pd, pe, gt=gt)
old_logp = torch.tensor([r[5] for r in records], device=device)
rewards = torch.tensor([r[6] for r in records], device=device)
discounts = torch.tensor([r[7] for r in records], device=device)
values = out['value'].squeeze(0)
vs, adv = vtrace(out['logp'].squeeze(0) - old_logp, discounts, rewards, values, bootstrap)
policy_loss = -(adv.detach() * out['logp'].squeeze(0)).mean()
value_loss = 0.5 * ((vs.detach() - values) ** 2).mean()
entropy_loss = -0.01 * out['entropy'].mean()
loss = policy_loss + value_loss + entropy_loss
optimizer.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_(policy.parameters(), 40.0)
optimizer.step()
print({'steps': T, 'terminated': terminated, 'truncated': truncated, 'loss': float(loss.item()), 'device': str(device)})
print('OK: single-notebook smoke training completed')

## E2E 마일스톤 실행
아다 셀은 M1~M5의 통합 경로를 실행합니다.

In [ ]:
# E2E M1-M5: batched device-resident environment and stale-actor V-trace learner
import copy, time
D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); torch.manual_seed(7)
N_ENVS, UNROLL, UPDATES, ACTOR_SYNC_EVERY = 128, 16, 12, 3
LANES, LANE_START, ALTAR_ENTITY, GAMMA = 7, 2, 9, .999

class BatchedJinHilla:
    def __init__(self, n, device): self.n,self.device=n,device; self.reset()
    def reset(self, done=None):
        if done is None:
            self.t=torch.zeros(self.n,dtype=torch.long,device=D); self.hp=torch.ones(self.n,device=D); self.green=torch.full((self.n,),5,dtype=torch.long,device=D); self.red=torch.zeros_like(self.green); self.lane=torch.full_like(self.green,3); self.altar=torch.full_like(self.green,-1); self.stun=torch.zeros_like(self.green); self.tele=torch.zeros(self.n,LANES,dtype=torch.long,device=D); self.impact=torch.zeros_like(self.tele); self.prev_interrupt=torch.zeros(self.n,device=D); self.prev_elapsed=torch.zeros_like(self.green)
        else:
            for x,v in [(self.t,0),(self.hp,1.),(self.green,5),(self.red,0),(self.lane,3),(self.altar,-1),(self.stun,0),(self.prev_interrupt,0.),(self.prev_elapsed,0)]: x[done]=v
            self.tele[done]=0; self.impact[done]=0
        return self.obs()
    def obs(self):
        e=torch.zeros(self.n,MAX_ENTITIES,ENTITY_DIM,device=D); m=torch.zeros(self.n,MAX_ENTITIES,dtype=torch.bool,device=D); m[:,:2]=True
        slots=torch.arange(LANE_START,LANE_START+LANES,device=D); e[:,1,EntityField.X]=self.lane.float()*2; e[:,slots,EntityField.X]=torch.arange(LANES,device=D).float()*2; e[:,slots,EntityField.IS_PLATFORM]=1; e[:,slots,EntityField.REACHABLE]=1; e[:,slots,EntityField.SAFETY]=(self.impact==0).float(); m[:,slots]=True
        has=self.altar>=0; e[has,ALTAR_ENTITY,EntityField.X]=self.altar[has].float()*2; e[has,ALTAR_ENTITY,EntityField.TYPE_ID]=2; e[has,ALTAR_ENTITY,EntityField.REACHABLE]=1; m[has,ALTAR_ENTITY]=True
        s=torch.zeros(self.n,SCALAR_DIM,device=D); s[:,ScalarIndex.PREV_INTERRUPT]=self.prev_interrupt; s[:,ScalarIndex.PREV_ELAPSED_NORM]=self.prev_elapsed.float()/(DELAY_BINS-1); s[:,ScalarIndex.HAZARD_LEVEL]=(self.impact>0).float().mean(1); s[:,ScalarIndex.BOSS_HP_NORM]=self.hp; s[:,ScalarIndex.GREEN_SKULLS]=self.green.float()/5; s[:,ScalarIndex.RED_SKULLS]=self.red.float()/5; s[:,ScalarIndex.ALTAR_PRESENT]=has.float(); s[:,ScalarIndex.TIME_REMAIN_NORM]=(1-self.t.float()/EPISODE_TICKS).clamp_min(0)
        return {'scalars':s,'entities':e,'entity_mask':m}
    def step(self,a):
        at,ad,skill,ptr=[x.long() for x in a]; r=torch.zeros(self.n,device=D); elapsed=torch.zeros(self.n,dtype=torch.long,device=D); intr=torch.zeros(self.n,dtype=torch.bool,device=D); term=torch.zeros_like(intr); active=torch.ones_like(intr); target=torch.where((ptr==ALTAR_ENTITY)&(self.altar>=0),self.altar,(ptr-LANE_START).clamp(0,LANES-1))
        for _ in range(DELAY_BINS):
            run=active&(elapsed<=ad)
            if not run.any(): break
            ok=run&(self.stun==0); evade=ok&(at==ActionType.EVADE)&(ptr!=NULL_ENTITY_INDEX); self.lane=torch.where(evade,target,self.lane)
            cast=ok&(at==ActionType.SKILL); dmg=.001+.00025*(skill.float()+1); self.hp=torch.where(cast,(self.hp-dmg).clamp_min(0),self.hp); r+=cast.float()*dmg*8
            self.t+=run.long(); elapsed+=run.long(); starts=run[:,None]&(self.tele==1); self.tele=torch.where(run[:,None]&(self.tele>0),self.tele-1,self.tele); self.impact=torch.where(starts,torch.full_like(self.impact,4),self.impact)
            hit=run&(self.impact[torch.arange(self.n,device=D),self.lane]>0)&(self.stun==0); r-=.15*hit.float(); self.green-=hit.long(); self.red+=hit.long(); self.stun=torch.where(hit,torch.full_like(self.stun,90),self.stun); intr|=hit; self.impact=torch.where(run[:,None]&(self.impact>0),self.impact-1,self.impact); cycle=run&((self.t%(8*TICKS_PER_SEC))==1); self.tele=torch.where(cycle[:,None],int(1.32*TICKS_PER_SEC)+3*torch.arange(LANES,device=D),self.tele); self.stun=torch.where(run&(self.stun>0),self.stun-1,self.stun)
            spawn=run&(self.red>=3)&(self.altar<0)&(self.green>0); self.altar=torch.where(spawn,self.t%LANES,self.altar); dead=self.green<=0; win=self.hp<=0; term|=dead|win; r-=10*dead.float(); r+=10*win.float(); active&=~(hit|dead|win)
        trunc=(self.t>=EPISODE_TICKS)&~term; self.prev_interrupt=intr.float(); self.prev_elapsed=elapsed.clamp_max(DELAY_BINS-1); return self.obs(),r,term,trunc,elapsed,intr

def batched_vtrace(log_ratio,discounts,rewards,values,bootstrap):
    with torch.no_grad():
        rho=log_ratio.exp().clamp(max=1); delta=rho*(rewards+discounts*torch.cat([values[:,1:],bootstrap[:,None]],1)-values); acc=torch.zeros_like(bootstrap); out=[]
        for i in range(delta.shape[1]-1,-1,-1): acc=delta[:,i]+discounts[:,i]*rho[:,i]*acc; out.append(acc)
        vs=values+torch.stack(out[::-1],1); adv=rho*(rewards+discounts*torch.cat([vs[:,1:],bootstrap[:,None]],1)-values)
    return vs,adv

# M1 regression: reset is safe; terminal telegraph tick causes an interrupt.
check=BatchedJinHilla(1,D); wait=(torch.tensor([ActionType.WAIT],device=D),torch.tensor([0],device=D),torch.tensor([0],device=D),torch.tensor([1],device=D)); assert not bool(check.step(wait)[-1]); check.tele.zero_(); check.tele[0,check.lane[0]]=1; assert bool(check.step(wait)[-1])
policy=Policy().to(D); actor=copy.deepcopy(policy).eval(); opt=torch.optim.Adam(policy.parameters(),3e-4); env=BatchedJinHilla(N_ENVS,D); obs=env.obs(); hidden=None; pt=torch.full((N_ENVS,1),ActionType.WAIT,dtype=torch.long,device=D); pd=torch.zeros_like(pt); pe=torch.zeros_like(pt); t0=time.perf_counter()
for update in range(UPDATES):
    if update%ACTOR_SYNC_EVERY==0: actor.load_state_dict(policy.state_dict())
    buf={k:[] for k in ('s','e','m','pt','pd','pe','at','ad','ak','ap','old','r','d')}; h0=None if hidden is None else tuple(x.detach().clone() for x in hidden)
    for _ in range(UNROLL):
        for k,x in [('s',obs['scalars']),('e',obs['entities']),('m',obs['entity_mask']),('pt',pt[:,0]),('pd',pd[:,0]),('pe',pe[:,0])]: buf[k].append(x)
        with torch.no_grad(): out=actor(obs['scalars'][:,None],obs['entities'][:,None],obs['entity_mask'][:,None],pt,pd,pe,hidden)
        a=out['type'][:,0],out['delay'][:,0],out['skill'][:,0],out['pointer'][:,0]; nxt,r,term,trunc,el,_=env.step(a)
        for k,x in zip(('at','ad','ak','ap'),a): buf[k].append(x)
        buf['old'].append(out['logp'][:,0]); buf['r'].append(r); buf['d'].append((~term).float()*(GAMMA**el.float())); hidden=tuple(x.detach() for x in out['hidden']); done=term|trunc
        for x in hidden: x[:,done]=0
        if done.any(): env.reset(done); nxt=env.obs()
        obs=nxt; pt,pd,pe=a[0][:,None],a[1][:,None],el.clamp_max(DELAY_BINS-1)[:,None]
    st=lambda k:torch.stack(buf[k],1); gt={'type':st('at'),'delay':st('ad'),'skill':st('ak'),'pointer':st('ap')}; out=policy(st('s'),st('e'),st('m'),st('pt'),st('pd'),st('pe'),h0,gt)
    with torch.no_grad(): bootstrap=policy(obs['scalars'][:,None],obs['entities'][:,None],obs['entity_mask'][:,None],pt,pd,pe,hidden)['value'][:,0]
    vs,adv=batched_vtrace(out['logp']-st('old'),st('d'),st('r'),out['value'],bootstrap); loss=-(adv.detach()*out['logp']).mean()+.5*((vs.detach()-out['value'])**2).mean()-.01*out['entropy'].mean(); opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(policy.parameters(),40); opt.step()
if D.type=='cuda': torch.cuda.synchronize()
print({'M1_regression':'pass','device':str(D),'transitions':N_ENVS*UNROLL*UPDATES,'SPS':round(N_ENVS*UNROLL*UPDATES/(time.perf_counter()-t0),1),'loss':float(loss),'actor_sync_every':ACTOR_SYNC_EVERY})
